In [1]:
from run_test import *
import os
import pandas as pd

/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [2]:
wavs = "/home/romolo/VT1/prog/streamlit_prog/outputs/xtts2_forward_iteration_run_large"
df_path = "/home/romolo/VT1/prog/streamlit_prog/outputs/csv/metadata_xtts2_forward_iteration_run_large.csv"

In [14]:
df = pd.read_csv(df_path)
df.head()

,ref_file,ref_path,target_file,target_path,output_file,output_path,WER,BLEU,speaker_similarity_an_ref,speaker_similarity_an_target,language,segment_quality_score,min_ref_seg,min_tar_seg
0,IT_416531,/home/romolo/VT1/coqui-tts/test_data/Dataset/r...,EN_1034,/home/romolo/VT1/coqui-tts/test_data/Dataset/r...,IT_416531_to_EN_1034.wav,/home/romolo/VT1/prog/streamlit_prog/outputs/x...,0.000000,1.000000,0.392832,0.005748,EN,"{'0': {'overall_quality': 0.6388320356607436, ...",0.136415,0.137334
1,IT_416531,/home/romolo/VT1/coqui-tts/test_data/Dataset/r...,EN_3259,/home/romolo/VT1/coqui-tts/test_data/Dataset/r...,IT_416531_to_EN_3259.wav,/home/romolo/VT1/prog/streamlit_prog/outputs/x...,0.020408,0.946459,0.400138,-0.005358,EN,"{'0': {'overall_quality': 0.5711261689662933, ...",0.160818,0.106322
2,IT_416531,/home/romolo/VT1/coqui-tts/test_data/Dataset/r...,EN_163,/home/romolo/VT1/coqui-tts/test_data/Dataset/r...,IT_416531_to_EN_163.wav,/home/romolo/VT1/prog/streamlit_prog/outputs/x...,0.043478,0.921059,0.388439,0.158307,EN,"{'0': {'overall_quality': 0.5392836870625615, ...",0.103714,0.148566
3,IT_416531,/home/romolo/VT1/coqui-tts/test_data/Dataset/r...,IT_416531,/home/romolo/VT1/coqui-tts/test_data/Dataset/r...,IT_416531_to_IT_416531.wav,/home/romolo/VT1/prog/streamlit_prog/outputs/x...,0.000000,1.000000,0.716549,0.651247,IT,"{'0': {'overall_quality': 0.4154738038778305, ...",0.423653,0.603509
4,IT_416531,/home/romolo/VT1/coqui-tts/test_data/Dataset/r...,EN_302,/home/romolo/VT1/coqui-tts/test_data/Dataset/r...,IT_416531_to_EN_302.wav,/home/romolo/VT1/prog/streamlit_prog/outputs/x...,0.000000,1.000000,0.301302,0.197358,EN,"{'0': {'overall_quality': 0.5949222615920007, ...",0.136585,0.230978


In [15]:
file_name = df["output_file"][0].split(".")[0]+"_0.wav"

In [16]:
df['output_path'][0].replace("IT_416531_to_EN_1034.wav", file_name)

'/home/romolo/VT1/prog/streamlit_prog/outputs/xtts2_forward_iteration_run_large/IT_416531_to_EN_1034_0.wav'

In [12]:
import json
import os
import pandas as pd
from tqdm import tqdm

def load_analysis_checkpoint(checkpoint_path):
    """Load analysis checkpoint"""
    if os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'r') as f:
                return json.load(f)
        except json.JSONDecodeError:
            tqdm.write("Checkpoint file corrupted, starting fresh")
            return {"completed": [], "results": []}
    else:
        return {"completed": [], "results": []}

def save_analysis_checkpoint(checkpoint_path, completed, results):
    """Save analysis checkpoint"""
    checkpoint = {
        "completed": completed,
        "results": results
    }
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
    with open(checkpoint_path, 'w') as f:
        json.dump(checkpoint, f, indent=2)

def analyze_wav(row):
    """
    alpha: float, between 0 and 1, controls the degree of anonymization. 1 full anonymization (take style embedding of reference voice), 0 means take the style embedding from the target (the input audio we want to anonymize)/n
    style: str, one of 'target', 'mixing', 'half_and_half'. Determines the style of voice conversion./n
    """
    import whisper
    import soundfile as sf
    ref_path = row['ref_path']
    ref_file = row["ref_file"]
    target_speaker_folder = row['target_file']
    target_path = row['target_path']
    lan = row['language']
    out_name = row["output_file"].split(".")[0]+"_0.wav"
    out_path = row['output_path'].replace(row["output_file"], out_name)

    # read wav
    anonymized_wav, sr = torchaudio.load(out_path)


    ref_speakers = os.listdir(ref_path)
    ref_speakers = [os.path.join(ref_path, spk) for spk in ref_speakers]
    target_speakers = os.listdir(target_path)
    target_speakers = [os.path.join(target_path, spk) for spk in target_speakers]


    # get whisper
    asr_model = whisper.load_model("base", device="cuda" if torch.cuda.is_available() else "cpu")
    
    _,segment_quality_score_value = segment_quality_score(
            asr_model,
            get_speaker_embedding(target_speakers[0]).to(device),
            get_speaker_embedding(ref_speakers).to(device),
            anonymized_wav,
            ecapa2
        )


    speaker_similarity_an_ref = calc_speaker_similarity(ref_speakers, out_path)
    speaker_similarity_an_target = calc_speaker_similarity(target_speakers[0], out_path)
    wer_score, bleu_score = calc_asr_wer_blue(target_speakers[0], out_path, asr_model)
    tqdm.write(f"Processed: {ref_file} + {target_speaker_folder} | Speaker Similarity: {speaker_similarity_an_ref:.4f} | WER: {wer_score} | BLEU: {bleu_score}")

 
    return {
        "ref_file": ref_file,
        "ref_path": ref_path,
        "target_file": target_speaker_folder,
        "target_path": target_path,
        "output_file": out_name,
        "output_path": out_path,
        "WER": wer_score,
        "BLEU": bleu_score,
        "speaker_similarity_an_ref": speaker_similarity_an_ref,
        "speaker_similarity_an_target": speaker_similarity_an_target,
        "language": lan,
        "segment_quality_score": segment_quality_score_value,
        "min_ref_seg": min([i['ref_sim'] for i in segment_quality_score_value.values()]),
        "min_tar_seg": max([i['target_sim'] for i in segment_quality_score_value.values()])
    }

def run_analysis_with_checkpointing(df, checkpoint_path="analysis_checkpoint.json"):
    """
    Run analysis with checkpointing
    """
    checkpoint = load_analysis_checkpoint(checkpoint_path)
    completed = set(checkpoint["completed"])
    results = checkpoint["results"]
    
    tqdm.write(f"Starting analysis. {len(completed)} already completed, {len(df) - len(completed)} remaining.")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Analyzing WAV files"):
        # Create unique identifier for this row
        row_id = f"{row['ref_file']}_{row['target_file']}"
        
        if row_id in completed:
            continue
            
        
        # Run your exact analyze_wav function
        result = analyze_wav(row)
        results.append(result)
        completed.add(row_id)
            
        # Save checkpoint after each analysis
        save_analysis_checkpoint(checkpoint_path, list(completed), results)
            
    
    tqdm.write(f"Analysis complete! Processed {len(results)} files total.")
    
    # Clean up checkpoint
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        tqdm.write("Checkpoint file removed")
    
    return results

# Usage:
# results = run_analysis_with_checkpointing(df, "my_analysis_checkpoint.json")
# results_df = pd.DataFrame(results)
# results_df.to_csv("analysis_results.csv", index=False)

In [ ]:
"""speakers = []
path = "/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/"
for i in os.listdir(path):
    speakers.append(path+i+'/'+os.listdir(path+i+'/')[0])"""




In [17]:
results = run_analysis_with_checkpointing(df, "./analysis_checkpoint.json")

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv("analysis_results.csv", index=False)

Starting analysis. 7596 already completed, 2403 remaining.


Analyzing WAV files:  60%|██████    | 6019/9999 [00:07<00:00, 60165.81it/s]

Processed: FR_412440 + FR_414792 | Speaker Similarity: 0.6151 | WER: 0.0 | BLEU: 1.0


Analyzing WAV files:  60%|██████    | 6019/9999 [00:11<00:00, 60165.81it/s]

Processed: IT_416492 + EN_3374 | Speaker Similarity: 0.4134 | WER: 0.0 | BLEU: 1.0


...

Processed: EN_6019 + EN_6529 | Speaker Similarity: 0.2984 | WER: 0.16129032258064516 | BLEU: 0.6552747921326864


Analyzing WAV files: 100%|█████████▉| 9998/9999 [1:57:54<00:02,  2.89s/it]

Processed: EN_6019 + EN_6019 | Speaker Similarity: 0.4257 | WER: 0.045454545454545456 | BLEU: 0.8791116082044841


Analyzing WAV files: 100%|██████████| 9999/9999 [1:57:54<00:00,  1.41it/s]


Analysis complete! Processed 9999 files total.
Checkpoint file removed


In [ ]:
def analyze_wav(row):
    """
    alpha: float, between 0 and 1, controls the degree of anonymization. 1 full anonymization (take style embedding of reference voice), 0 means take the style embedding from the target (the input audio we want to anonymize)/n
    style: str, one of 'target', 'mixing', 'half_and_half'. Determines the style of voice conversion./n
    """
    import whisper
    import soundfile as sf
    ref_path = row['ref_path']
    ref_file = row["ref_file"]
    target_speaker_folder = row['target_file']
    target_path = row['target_path']
    lan = row['language']
    out_name = row["output_file"].split(".")[0]+"_0.wav"
    out_path = row['output_path'].replace(row["output_file"], out_name)

    # read wav
    anonymized_wav, sr = torchaudio.load(out_path)


    ref_speakers = os.listdir(ref_path)
    ref_speakers = [os.path.join(ref_path, spk) for spk in ref_speakers]
    target_speakers = os.listdir(target_path)
    target_speakers = [os.path.join(target_path, spk) for spk in target_speakers]


    # get whisper
    asr_model = whisper.load_model("base", device="cuda" if torch.cuda.is_available() else "cpu")
    
    _,segment_quality_score_value = segment_quality_score(
            asr_model,
            get_speaker_embedding(target_speakers[0]).to(device),
            get_speaker_embedding(ref_speakers).to(device),
            anonymized_wav,
            ecapa2
        )



    speaker_similarity_an_ref = calc_speaker_similarity(ref_speakers, out_path)
    speaker_similarity_an_target = calc_speaker_similarity(target_speakers[0], out_path)
    wer_score, bleu_score = calc_asr_wer_blue(target_speakers[0], out_path, asr_model)
    tqdm.tqdm.write(f"Processed: {ref_file} + {target_speaker_folder} | Speaker Similarity: {speaker_similarity_an_ref:.4f} | WER: {wer_score} | BLEU: {bleu_score}")

 
    return {
        "ref_file": ref_file,
        "ref_path": ref_path,
        "target_file": target_speaker_folder,
        "target_path": target_path,
        "output_file": out_name,
        "output_path": out_path,
        "WER": wer_score,
        "BLEU": bleu_score,
        "speaker_similarity_an_ref": speaker_similarity_an_ref,
        "speaker_similarity_an_target": speaker_similarity_an_target,
        "language": lan,
        "segment_quality_score": segment_quality_score_value,
        "min_ref_seg": min([i['ref_sim'] for i in segment_quality_score_value.values()]),
        "min_tar_seg": max([i['target_sim'] for i in segment_quality_score_value.values()])
    }

In [ ]:
collect_metadata = []

for index,df_row in tqdm.tqdm(df.iterrows()):
    metadata = analyze_wav(df_row)
    collect_metadata.append(metadata)
    
